In [2]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [3]:
import os
os.environ["PATH"] += os.pathsep + '/home/icb/mostafa.shahhosseini/miniconda3/envs/lupien/bin'
import pybedtools
import numpy as np
import pandas as pd
from tqdm import tqdm
import chromATAC as ca
from chromATAC.integrated import IntData
from functools import reduce
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')
from concurrent.futures import ProcessPoolExecutor
from chromATAC.al import mann_whitney_test, wilcoxon_test
from scipy.stats import ks_2samp

/home/icb/mostafa.shahhosseini/miniconda3/envs/lupien/lib/python3.11/site-packages/cudf/utils/gpu_utils.py:62: UserWarning: Failed to dlopen libcuda.so.1
  warnings.warn(str(e))


In [4]:
parent_dir = '/home/icb/mostafa.shahhosseini/data/lupien/'

In [5]:
os.chdir(parent_dir)

In [6]:
def compare_category_pairs(df, 
                           categories, 
                           group_col='Group', 
                           target_col='peak_count',
                           test='mann_whitney',
                           nz=False
                           ):
    all_categories = df[group_col].unique()
    other_categories = [cat for cat in all_categories if cat not in categories]
    pvalues = {}
    for cat1 in categories:
        pvalues[cat1] = []
        for cat2 in other_categories:
            s1 = df[df[group_col]==cat1].loc[:, target_col].values
            s2 = df[df[group_col]==cat2].loc[:, target_col].values
            if nz:
                s1 = s1[s1!=0]
                s2 = s2[s2!=0]
            if test=='mann_whitney':
                p_value = mann_whitney_test(s1, s2)
            if test=='ks':
                _, p_value = ks_2samp(s1, s2)
            pvalues[cat1].append(p_value)
    pval_df = pd.DataFrame(pvalues)
    pval_df.index=other_categories
    return pval_df

In [7]:
def compare_groupings(df, nz=False, test='mann_whitney', tqdm_active=False):
    groups = [i for i in df.columns if i.startswith('group')]
    pval_dfs = {}
    for g in tqdm(groups) if tqdm_active else groups:
        n = int(g.replace('groups', ''))
        pval_dfs[g] = []
        for i in range(1, n+1):
            group_ct = df['Group'][df[g]==i].unique()
            pval = compare_category_pairs(df, [i], group_col=g, nz=nz, test=test)
            pval_dfs[g].append(pval)
        pval_dfs[g] = pd.concat(pval_dfs[g], axis=1).sort_index()
    return pval_dfs

In [8]:
def kde_groupped_plot(d):
    fig, axs=plt.subplots(23, 1, figsize=(12, 8), sharex=True)
    for i, g in enumerate(d['Group'].unique()):
        ax = axs.flatten()[i]
        sns.kdeplot(d[d['Group']==g]['peak_count'], ax=ax, fill=True)
        ax.set_title(g, fontsize=8)
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
        ax.spines['bottom'].set_visible(False)
        ax.spines['left'].set_visible(False)
    plt.show()

In [9]:
def pivot_df(df):
    df['region'] = df.apply(lambda row:f"{row['TE']}|{row['chrom']}:{row['start']}>{row['end']}", axis=1)
    return df.pivot(index='sample', columns='region', values='peak_count')

--------

In [10]:
grouping_tables = []
for f in os.listdir('./new-TCGA/groupings/'):
    if f.endswith('.csv'):
        with open(os.path.join('./new-TCGA/groupings/', f)) as t:
            d = pd.read_csv(t)
            grouping_tables += [d]

In [11]:
groups = pd.concat([i.iloc[:, 4:] for i in grouping_tables], axis=1)
groups = pd.concat([d.iloc[:, [2, 3]], groups], axis=1)

In [13]:
tes = os.listdir('./tes/')

In [21]:
dt = pd.read_csv(f'./tes/{tes[0]}', sep='\t', index_col=0)

great = pd.DataFrame(columns=dt.columns)

In [22]:
pd.concat([great, dt])

,0020e896-eb44-4d34-b3fc-bf55df812733,00919664-1563-46f2-a9de-290ba63f8ca0,00a595c5-0565-4289-878a-80ff4ba89d97,017ee2ce-79cb-475f-a497-46a8c57b8dd4,01a439d2-f4ba-4c1f-9a13-4d8872058b08,02f56cb3-dc15-4dbd-a508-cbb14154e1bd,0338ad51-21c2-4607-a60e-099fbe0693ba,03c50711-9e22-4275-90aa-b6cd56584437,03d0944c-b853-4b09-bd82-5db368275033,03e41ffe-e1ac-4575-ace5-c376628c890e,...,fbd18a1c-d602-488f-a072-91c61919a171,fbf3a88f-5308-4234-874d-8a99e971efaa,fc13e8f3-35e3-40c3-8376-3af6f2bccb3a,fc3fe6ee-8cec-4e29-90c2-0ceebbf66adc,fc9a595e-2077-49d8-a8a5-f026f8a7345e,feccc9ed-3000-4549-be59-9604929f5b23,ff4d2e83-f6aa-4e98-ae75-11a97a7d8253,ff883a34-10d4-4320-b37c-16fcdd637dbc,ff9cbc57-60f4-45c7-82f6-7670199ba28c,ffb8a92c-4f13-4b72-abee-d6f5c5d8f66f
LTR_ERV1_LTR1E|chr10:115449022.0>115449797.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
LTR_ERV1_LTR1E|chr10:116169324.0>116170128.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
LTR_ERV1_LTR1E|chr10:126691253.0>126692032.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
LTR_ERV1_LTR1E|chr10:18887638.0>18888417.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
LTR_ERV1_LTR1E|chr10:3185129.0>3185893.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
LTR_ERV1_LTR1E|chrX:19069419.0>19070199.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
LTR_ERV1_LTR1E|chrX:29467421.0>29467620.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
LTR_ERV1_LTR1E|chrX:46262809.0>46263639.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,...,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
LTR_ERV1_LTR1E|chrX:48084820.0>48085640.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [ ]:
te_cancer_encyclopedia = pd.read_csv('./te_cancer_encyclopedia2.tsv', sep='\t', index_col=0)

In [12]:
samples = te_cancer_encyclopedia['sample'].unique()
tes_samples = pd.DataFrame({i:[] for i in samples})

tes = te_cancer_encyclopedia['TE'].unique()

In [ ]:
for t in tes:
    g = te_cancer_encyclopedia.loc[te_cancer_encyclopedia['TE']==t]
    g['region'] = g.apply(lambda row:f"{row['TE']}|{row['chrom']}:{row['start']}>{row['end']}", axis=1)
    g = g.loc[:, ['region', 'peak_count', 'sample']]
    g = g.pivot(columns='region', index='sample', values='peak_count')
    g = g.T
    g.to_csv(f'./tes/{t}.tsv', sep='\t')
    tes_samples  = pd.concat([tes_samples, g])
tes_samples.to_csv('TEs_TCGA.tsv', sep='\t')

In [13]:
import concurrent.futures

In [14]:
# Function to process each TE
def process_te(t):
    g = te_cancer_encyclopedia.loc[te_cancer_encyclopedia['TE'] == t]
    g['region'] = g.apply(lambda row: f"{row['TE']}|{row['chrom']}:{row['start']}>{row['end']}", axis=1)
    g = g.loc[:, ['region', 'peak_count', 'sample']]
    g = g.pivot(columns='region', index='sample', values='peak_count')
    g = g.T
    g.to_csv(f'./tes/{t}.tsv', sep='\t')
    return g

# Use ProcessPoolExecutor to parallelize the processing
with concurrent.futures.ProcessPoolExecutor() as executor:
    results = list(executor.map(process_te, tes[:3]))

# Concatenate the results into a single DataFrame
for result in results:
    tes_samples = pd.concat([tes_samples, result])

In [ ]:
# Save the final combined DataFrame
tes_samples.to_csv('TEs_TCGA.tsv', sep='\t')

In [11]:
te_cancer_encyclopedia = pd.DataFrame({'chrom':[], 'start':[], 'end':[], 'peak_count':[], 'sample':[], 'TE':[]})

In [17]:
# for ft in tqdm(os.listdir('./new-TCGA/TE.intersection/')):
#     if ft.endswith('.tsv'):
#         df = pd.read_csv(os.path.join('./new-TCGA/TE.intersection/', ft), index_col=0, sep='\t')
#         df = df[df['chrom'].apply(lambda x: x in ca.info.CHROMOSOMES['names'])]
#         df['sample'] = df['sample'].apply(lambda x: x.split('_peak')[0])
#         df.set_index((['chrom', 'start', 'end']), inplace=True)
#         f = df.groupby(['chrom', 'start', 'end']).sum()['peak_count']
#         fix = f[f!=0].index
#         df = df.loc[fix, :]
#         df.reset_index(inplace=True)
#         df['TE'] = ft.replace('.tsv', '')
#         te_cancer_encyclopedia = pd.concat([df, te_cancer_encyclopedia])

In [25]:
hp = pivot_df(h)

In [28]:
h.to_csv('hj.csv')

In [ ]:
te_cancer_encyclopedia = te_cancer_encyclopedia.merge(groups, left_on='sample', right_on='SampleName').drop(columns='SampleName').rename({'Group':'cancer type'}, axis=1)

In [ ]:
te_cancer_encyclopedia.to_csv('te_cancer_encyclopedia.tsv', sep='\t')

In [ ]:
te_cancer_encyclopedia

In [30]:
h

,chrom,start,end,peak_count,sample,TE,region
7580991,chr12,110116624.0,110116935.0,0.0,ffb8a92c-4f13-4b72-abee-d6f5c5d8f66f,SINE_Alu_AluJb,SINE_Alu_AluJb|chr12:110116624.0>110116935.0
196129,chr1,186243602.0,186243925.0,0.0,e11d6396-dd5a-4f94-aaf8-119edb3904a0,LTR_ERVL-MaLR_MLT1J,LTR_ERVL-MaLR_MLT1J|chr1:186243602.0>186243925.0
2585695,chr12,57834840.0,57835340.0,0.0,f00263d8-f97f-494a-bc8e-1700fc81c2a3,SINE_Alu_AluSp,SINE_Alu_AluSp|chr12:57834840.0>57835340.0
1162274,chr16,8952385.0,8952556.0,0.0,d4221927-7de2-4f7d-a677-a5caefb5f28a,LTR_ERVL-MaLR_MLT1D,LTR_ERVL-MaLR_MLT1D|chr16:8952385.0>8952556.0
182900,chr1,228887461.0,228887950.0,0.0,8c128110-4bf4-4ede-9361-c39be9c2f275,LINE_L1_L1MC5a,LINE_L1_L1MC5a|chr1:228887461.0>228887950.0
...,...,...,...,...,...,...,...
7764370,chr16,11230903.0,11231082.0,0.0,fa47df40-1616-454f-a926-29a0b3be68a1,SINE_Alu_AluY,SINE_Alu_AluY|chr16:11230903.0>11231082.0
19571158,chr9,125336510.0,125336647.0,0.0,9fbbc8b0-b69d-45c1-8f10-6761352bcf8d,SINE_Alu_AluJo,SINE_Alu_AluJo|chr9:125336510.0>125336647.0
76017,chr14,96102102.0,96102517.0,0.0,63fec5bd-3184-4974-a1b7-a43b82e1ae9c,LTR_ERVL_MLT2C2,LTR_ERVL_MLT2C2|chr14:96102102.0>96102517.0
38455937,chr8,144535770.0,144535882.0,0.0,a908353f-9756-4502-bc4b-7427604ec70c,SINE_MIR_MIR,SINE_MIR_MIR|chr8:144535770.0>144535882.0


In [31]:
memory_per_row = h.memory_usage(deep=True).sum() / len(h)
print(f"Memory per row: {memory_per_row / 1024:.2f} KB")


Memory per row: 0.35 KB


In [ ]:
import os
import dask.dataframe as dd
import pandas as pd
from dask.distributed import Client
from dask import delayed


In [33]:
# Assume you want to use only 50% of available memory for processing
safe_memory_usage = available_memory * 0.5
rows_per_chunk = safe_memory_usage // memory_per_row
blocksize = rows_per_chunk * h.memory_usage(deep=True).sum() / len(h)
print(f"Rows per chunk: {rows_per_chunk}")
print(f"Block size: {blocksize / (1024 ** 2):.2f} MB")


Rows per chunk: 570114986.0
Block size: 194830.90 MB


In [35]:
import os